<a href="https://colab.research.google.com/github/liwenxi818/git-project/blob/main/2%EC%A3%BC%EC%B0%A8_%EA%B3%BC%EC%A0%9C_20251920_LI_WENXI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2주차 과제 (MLP) - 50 points

## Multi-Layer Perceptron with MNIST handwritten digits classification
## Forward Propagation 구현하기

## - Module Import

In [50]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from torchvision import datasets
import torchvision.transforms as transforms

## - 딥러닝 모델을 설계할 때 활용하는 장비 확인

In [51]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Using PyTorch version:', torch.__version__, ' Device:', device)

Using PyTorch version: 2.10.0+cpu  Device: cpu


## - MNIST 데이터 다운로드 (Train data와 Test data 분리하기)

In [52]:
BATCH_SIZE = 32

train_data = datasets.MNIST('./data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST('./data', train=False, download=True, transform=transforms.ToTensor())

train_loader = torch.utils.data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = torch.utils.data.DataLoader(dataset=test_data, batch_size=BATCH_SIZE, shuffle=False)

## - 첫번째 batch 데이터의 크기와 타입을 확인하기

In [53]:
for (X_train, y_train) in train_loader:
    print('X_train:', X_train.size(), 'type:', X_train.type())
    print('y_train:', y_train.size(), 'type:', y_train.type())
    break

X_train: torch.Size([32, 1, 28, 28]) type: torch.FloatTensor
y_train: torch.Size([32]) type: torch.LongTensor


## Loss 계산하기
   ### forward propagation을 계산하는 함수 구현하기
   
   1) input layer (입력층), hidden layer (은닉층), output layer (출력층) 으로 이루어진 모델을 이용하시오.

   2) 하나의 hidden layer (은닉층)만 이용 - 은닉층의 개수는 100개로 하시오.

   3) 모든 것은 tensor 계산으로만 하시오.


## ReLU, one_hot_encoding, softmax, cross_entropy 계산하기

아래 코드는 각 함수가 맞는지 확인하기 위해서 만든 임의의 값임. 각 함수가 작동을 잘하는지 확인해 보시오.

In [54]:
test_data = torch.tensor([[1,-2,-4, 2, 5, 6, -3, -5, 0, 2],[2, -3, 4, 3, -1, -4, 3, 5, 2, -3]])
true_label = torch.tensor([5,7])
false_label = torch.tensor([1,0])

# Problem 1 (10 points)

## ReLU 함수를 구현하시오.

In [55]:
def ReLU_func(outputs):
    zero_tensor = torch.zeros(outputs.shape)
    final_outputs = torch.maximum(outputs, zero_tensor)

    return final_outputs

ReLU 함수가 맞는지 test_data를 이용하여 맞추어 보시오. 아래 함수의 결과는 어떻게 예상되는가?

In [56]:
ReLU_func(test_data)

tensor([[1., 0., 0., 2., 5., 6., 0., 0., 0., 2.],
        [2., 0., 4., 3., 0., 0., 3., 5., 2., 0.]])

# Problem 2 (10 points)

## one-hot encoding 함수를 구현하시오.

In [57]:
def one_hot_encoding(label):

    encode = torch.eye(10)
    one_hot_outputs = encode[label]

    return one_hot_outputs

one_hot_encoding 함수가 맞는지 true_label, false_label을 이용하여 맞추어 보시오. 아래 함수 결과는 어떻게 예상되는가?

In [58]:
# 검증을 위해 아래 4줄을 사용하면 됨.
tl = one_hot_encoding(true_label)
print(tl)
fl = one_hot_encoding(false_label)
print(fl)

# 나중을 위해서 사용될 코드임.
close_tl = tl.clone()
close_tl[[0,1],true_label] -= 0.001
close_tl[false_label] += 0.0001
print(close_tl)

tensor([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])
tensor([[0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
tensor([[1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 9.9910e-01,
         1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04],
        [1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04, 1.0000e-04,
         1.0000e-04, 9.9910e-01, 1.0000e-04, 1.0000e-04]])


# Problem 3 (10 points)

## softmax 함수를 구현하시오.

In [59]:
def softmax(outputs):
    numerator = torch.exp(outputs)
    denominator = torch.sum(numerator, dim=1).view(-1,1)
    softmax = numerator/denominator

    return softmax

softmax 함수가 맞는지 test_data를 이용하여 맞추어 보시오. 아래 함수의 결과는 어떻게 예상되는가?

In [60]:
a = softmax(test_data)
print(a)
torch.sum(a,axis=1)

tensor([[4.7643e-03, 2.3720e-04, 3.2102e-05, 1.2951e-02, 2.6012e-01, 7.0709e-01,
         8.7262e-05, 1.1810e-05, 1.7527e-03, 1.2951e-02],
        [2.8590e-02, 1.9264e-04, 2.1126e-01, 7.7716e-02, 1.4234e-03, 7.0868e-05,
         7.7716e-02, 5.7425e-01, 2.8590e-02, 1.9264e-04]])


tensor([1.0000, 1.0000])

# Problem 4 (10 points)

## cross entropy 오차 함수를 구현하시오.

In [61]:
def cross_entropy(outputs, labels):
    return -torch.sum(labels*torch.log(outputs))

### cross_entropy 함수가 잘 구현되었는지 test_data를 이용하여 맞추어 보시오.
아래 함수의 결과는 어떻게 예상되는가? tl, fl, close_tl을 이용하여 각각의 cross entropy를 구하고 그 값이 맞는지 확인하시오.

In [62]:
ideal_result = cross_entropy(close_tl, tl)
non_ideal_result = cross_entropy(close_tl,fl)

print(ideal_result)
print(non_ideal_result)

true_result = cross_entropy(a,tl)
false_result = cross_entropy(a,fl)

print(true_result)
print(false_result)

tensor(0.0018)
tensor(18.4207)
tensor(0.9013)
tensor(11.9013)


# Problem 5 (10 points)

## Forward propagation 과정을 구현하시오.

In [63]:
def forward_pass(train_loader):
    for batch_idx, (image, label) in enumerate(train_loader):

        image = image[0] # 1개의 입력 이미지와 해당 label을 이용하여 foward propagation를 계산
        label = label[0]

        image = image.view(-1, 28 * 28)
        print(image.size())

        # Weight(가중치)를 초기화 (torch rand 함수 이용, 도중에 빼기 0.5를 하여 함수값이 -0.5~0.5 사이로 만드시오.)
        W_ih = torch.rand(784,100)-0.5
        W_ho = torch.rand(100,10)-0.5

        # 첫번째 layer의 값 계산하기
        outputs = torch.mm(image, W_ih)

        # 첫번째 layer의 결과 값(outputs)을 ReLU 함수에 적용하기
        outputs = ReLU_func(outputs)


        # 출력 layer의 값 계산하기
        outputs = torch.mm(outputs, W_ho)
        print(outputs.size())

        # softmax 적용하기
        softmaxed_outputs = softmax(outputs)
        print(softmaxed_outputs.size())

        # label 값을 one_hot 형태로 변환하기
        expected_outputs = one_hot_encoding(label)

        # loss 값 계산하기
        loss = cross_entropy(softmaxed_outputs, expected_outputs)
        print(loss)

        break

In [64]:
forward_pass(train_loader) # 최종 결과값이 맞다면 아래 Markdown cell과 같은 값이 나오게 됩니다. 단, 4번째 줄의 값은 변동 가능합니다.

torch.Size([1, 784])
torch.Size([1, 10])
torch.Size([1, 10])
tensor(14.1286)


### 출력 결과 예:
torch.Size([1, 784])

torch.Size([1, 100])

torch.Size([1, 10])

tensor([13.0287])